### SmallTalk 1: Data, RAG and Baselines
**Can Agentic RAG and LoRA-Based DPO Help 4B Models Close the Gap with Larger Models in Meeting Summarization?**

This notebook loads QMSum, builds the retriever, measures retrieval quality against the gold spans, and runs the baselines for every model: zero-shot, standard RAG and agentic RAG.

### Setup
On Colab, choose **Runtime > Change runtime type > T4 GPU** first. Upload `smalltalk.py`, `config.py` and `requirements.txt` to `MyDrive/smalltalk/` (or keep them next to this notebook in VS Code).

In [ ]:
import os, sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/smalltalk'  # results are saved on Drive so a disconnect does not lose them
else:
    WORK = os.getcwd()
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, WORK)   # so the notebook can import smalltalk.py and config.py
print('Working folder:', WORK)
for f in ['smalltalk.py', 'config.py', 'requirements.txt']:
    assert os.path.exists(f), f'Put {f} in the working folder first'

In [ ]:
%pip install -q -r requirements.txt

### Config
All settings (models, hyperparameters, prompts) are in `config.py`. Edit that file, save it, and run this cell again. To change a setting for this notebook only, set it in a cell after this one.

`RUN_ON` sets where the notebook runs: `'local'` (6GB laptop), `'colab'` (T4), `'server'` (Fanshawe) or `'auto'` to detect it. It only changes the batch size and whether the 8B/9B models and training are run, so the results stay comparable.

`DOWNLOADS` is the one folder that holds everything downloaded: the dataset in `QMSum/` and the base models in `models/`. What is already there is reused, and only what is missing is downloaded.

In [ ]:
import json, glob, importlib
import pandas as pd
import config, smalltalk as st
importlib.reload(config); importlib.reload(st)   # picks up edits to config.py without a kernel restart
from config import *                              # every setting lives in config.py

st.set_seed(SEED)
os.makedirs('runs', exist_ok=True); os.makedirs('results', exist_ok=True)

PROFILE = st.run_profile(RUN_ON)
BATCH = PROFILE['batch_size']
# reuses the dataset and models that are already in the folder, downloads what is missing
st.setup_downloads(DOWNLOADS or PROFILE['downloads'])
# only needed if a model asks you to accept its terms on Hugging Face (keep it after setup_downloads)
# from huggingface_hub import login; login()

train_meetings, val_meetings, test_meetings = (st.load_split(s) for s in ['train', 'val', 'test'])
meetings_by_id = {m['id']: m for m in train_meetings + val_meetings + test_meetings}
test_examples = st.test_subset(st.build_examples(test_meetings), N_TEST)
print('Test questions used:', len(test_examples), '| K =', K, '| 4-bit:', LOAD_4BIT)

### 1. Data
QMSum has 232 meetings (product design, academic and parliament committee meetings). Each meeting has general questions ("Summarize the whole meeting") and specific questions with gold spans that mark which speaker turns hold the answer.

In [ ]:
for name, ms in [('train', train_meetings), ('val', val_meetings), ('test', test_meetings)]:
    ex = st.build_examples(ms)
    words = [sum(len(t['text'].split()) for t in m['turns']) for m in ms]
    print(f"{name}: {len(ms)} meetings, {len(ex)} questions "
          f"({sum(e['qtype'] == 'general' for e in ex)} general), median meeting length {int(pd.Series(words).median())} words")

ex = test_examples[0]
print('\nExample question:', ex['query'])
print('Gold summary:', ex['answer'][:300])

### 2. Retrieval
Meetings are split into chunks of about 200 words, embedded with `bge-small-en-v1.5` and stored in one FAISS index per meeting (LangChain). A chunk counts as relevant if it overlaps a gold span. Only specific questions have gold spans, so these metrics use those.

In [ ]:
retriever = st.Retriever()
ret = st.retrieval_metrics(test_examples, meetings_by_id, retriever, ks=(3, 5, 8, 10))
ret_df = pd.DataFrame({k: [ret[f'Hit@{k}'], ret[f'Recall@{k}'], ret[f'Precision@{k}']] for k in (3, 5, 8, 10)},
                      index=['Hit@k', 'Recall@k', 'Precision@k']).round(3)
print('MRR:', round(ret['MRR'], 3), '| questions:', ret['n_questions'])
os.makedirs('results', exist_ok=True)
ret_df.to_csv('results/retrieval.csv')
ret_df

### 3. The agent tools
The agent can call four LangChain tools when the starting chunks are not enough. Here is what each one returns for one meeting.

In [ ]:
ex = next(e for e in test_examples if e['qtype'] == 'specific')
m = meetings_by_id[ex['meeting_id']]
tools = st.make_tools(m, retriever)
print('Question:', ex['query'], '\n')
print(tools['list_topics'].invoke(''), '\n')
print(tools['search_transcript'].invoke(ex['query'])[:600], '\n')
print(tools['get_topic_segment'].invoke('1')[:400])

### 4. Baselines
Each model is loaded once, runs its modes, then is unloaded to free GPU memory. Results are written line by line to `runs/`, so if Colab disconnects just run the cell again and it continues.

- **zeroshot**: first 6,000 tokens of the transcript, no retrieval
- **rag**: top-k chunks (one chunk per topic for general questions)
- **agent**: starts with the same chunks as rag, then the model decides to answer right away or to call tools to read more (ReAct loop, max 4 actions)

Rough time on a T4 for 150 questions: zeroshot/rag 20 to 40 min, agent 1 to 2 h per model. The 8B/9B models only fit in 4-bit and are best run on Colab, not a 6GB laptop GPU.

In [ ]:
def run_model(key, model_id, modes, adapter=None, stage='base'):
    llm = st.LLM(model_id, adapter=adapter, load_4bit=LOAD_4BIT)
    for mode in modes:
        st.run_experiment(llm, mode, test_examples, meetings_by_id, retriever, f'runs/{key}_{stage}_{mode}.jsonl', k=K, batch_size=BATCH)
    llm.unload()

# quick check before the long runs: one answer from the first small model
llm = st.LLM(SMALL_MODELS['qwen3_4b'], load_4bit=LOAD_4BIT)
ctx, _ = st.rag_context(m, ex, retriever, k=K)
print(llm.chat(st.summary_messages(ctx, ex['query'])))
print('\nAgent:', st.run_agent(llm, m, ex, retriever, k=K))
llm.unload()

In [ ]:
for key, model_id in SMALL_MODELS.items():
    run_model(key, model_id, ['zeroshot', 'rag', 'agent'])

In [ ]:
# the 8B/9B models do not fit in 6GB, so the local profile skips them
if PROFILE['big_models']:
    for key, model_id in BIG_MODELS.items():
        run_model(key, model_id, ['rag', 'agent'])
else:
    print('Skipping the big models here, run this cell on Colab or the server')

### 5. Quick look
ROUGE only here (fast). Full scoring with BERTScore and faithfulness is in notebook 3.

In [ ]:
rows = [st.evaluate_run(p, use_bertscore=False) for p in sorted(glob.glob('runs/*_base_*.jsonl'))]
pd.DataFrame(rows).round(3)

In [ ]:
# one agent run that used tools, to see which ones it picked
agent_rows = st.read_jsonl(sorted(glob.glob('runs/*_base_agent.jsonl'))[0])
r = next((x for x in agent_rows if x['trace']), agent_rows[0])
print('Question:', r['query'])
for step in r['trace']:
    print('  ->', step['tool'], '|', step['input'])
print('\nAnswer:', r['pred'])
print('\nGold:', r['answer'])